# IGSN Catalog

This notebook builds a catalog of every sample (IGSN) with data in the AIMDL collection. It queries `/aimdl/partition` once per data type, the same way `03_multimodal_sample_discovery` does for two types, and lays the result out as one table: a row per IGSN, a column per data type.

It answers:

- How much data each IGSN has, and from which instruments
- Which data types each IGSN is missing
- Which data types have nothing at all

In [10]:
import json
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
import requests
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

sys.path.insert(0, str(Path.cwd().parent.parent / 'src'))
from aimdl_examples import get_girder_client

## Query partitions for every data type

`/aimdl/datatype` lists the types (including a `None` entry, which is skipped). Each partition key is `IGSN//Date`, so counting keys per IGSN gives the number of dated sessions an IGSN has for that type.

In [11]:
with requests.Session() as session:
    gc = get_girder_client(session=session)

    data_types = [dt for dt in gc.get('aimdl/datatype') if dt]

    partitions = {}
    for dt in data_types:
        keys = gc.get('aimdl/partition', parameters={'dataType': dt})
        # Parse partition keys to extract IGSN (before the '//' separator)
        partitions[dt] = Counter(key.split('//')[0] for key in keys)

for dt, igsns in partitions.items():
    print(f'{dt:<24} {len(igsns):>5} IGSNs  {sum(igsns.values()):>5} partitions')

nmd_derived                  0 IGSNs      0 partitions
nmd_project                144 IGSNs    166 partitions
nmd_raw                    142 IGSNs    164 partitions
pdv_alpss_output           361 IGSNs    385 partitions
pdv_alpss_result           361 IGSNs    385 partitions
pdv_experiment_log         330 IGSNs    357 partitions
pdv_trace                  361 IGSNs    386 partitions
xrd_calibrant_derived        0 IGSNs      0 partitions
xrd_calibrant_raw           10 IGSNs     18 partitions
xrd_derived                419 IGSNs   1890 partitions
xrd_metadata               397 IGSNs   1908 partitions
xrd_raw                    420 IGSNs   1892 partitions
xrd_visualization          396 IGSNs   2641 partitions
xrf_raw                    223 IGSNs   1249 partitions


## Build the catalog

One row per IGSN that appears in any partition, one column per data type. Each cell is the number of partitions; 0 means the IGSN has nothing of that type.

In [12]:
INSTRUMENT = {'pdv': 'HELIX', 'xrd': 'MAXIMA', 'xrf': 'MAXIMA', 'nmd': 'SPHINX'}

catalog = (
    pd.DataFrame(partitions)
    .reindex(columns=data_types)
    .fillna(0)
    .astype(int)
    .sort_index()
)
instrument_of = {dt: INSTRUMENT[dt.split('_')[0]] for dt in data_types}

print(f'{len(catalog)} IGSNs x {len(data_types)} data types')

513 IGSNs x 14 data types


## Data types with nothing

A data type with no partitions has no files tied to any IGSN. It's still listed by `/aimdl/datatype`, but querying it returns nothing.

In [13]:
coverage = pd.DataFrame({
    'instrument': pd.Series(instrument_of),
    'IGSNs with data': (catalog > 0).sum(),
    'IGSNs without': (catalog == 0).sum(),
    'partitions': catalog.sum(),
})
coverage['coverage'] = (coverage['IGSNs with data'] / len(catalog)).map('{:.0%}'.format)
coverage.sort_values(['instrument', 'IGSNs with data'], ascending=[True, False])

,instrument,IGSNs with data,IGSNs without,partitions,coverage
pdv_alpss_output,HELIX,361,152,385,70%
pdv_alpss_result,HELIX,361,152,385,70%
pdv_trace,HELIX,361,152,386,70%
pdv_experiment_log,HELIX,330,183,357,64%
xrd_raw,MAXIMA,420,93,1892,82%
xrd_derived,MAXIMA,419,94,1890,82%
xrd_metadata,MAXIMA,397,116,1908,77%
xrd_visualization,MAXIMA,396,117,2641,77%
xrf_raw,MAXIMA,223,290,1249,43%
xrd_calibrant_raw,MAXIMA,10,503,18,2%


## How much data each IGSN has

`types` is how many of the data types an IGSN has, and `instruments` names the instruments it went through. An IGSN with a single type usually has raw data the pipeline has not processed yet.

In [14]:
has_type = catalog > 0
by_instrument = has_type.T.groupby(instrument_of).any().T

summary = pd.DataFrame({
    'types': has_type.sum(axis=1),
    'partitions': catalog.sum(axis=1),
    'instruments': by_instrument.apply(lambda row: ' + '.join(row.index[row]), axis=1),
})

print('IGSNs by instrument combination:')
print(summary['instruments'].value_counts().to_string())
print()
print('IGSNs by number of data types present:')
print(summary['types'].value_counts().sort_index().to_string())

IGSNs by instrument combination:
instruments
HELIX + MAXIMA             160
MAXIMA                     132
HELIX + MAXIMA + SPHINX    123
HELIX                       76
HELIX + SPHINX              11
MAXIMA + SPHINX              7
SPHINX                       4

IGSNs by number of data types present:
types
1       3
2      27
3      35
4      98
5      47
6      19
7      14
8      44
9     110
10     55
11     61


## Missing data per instrument

Each instrument has a core set of types that every measured sample should end up with: the raw file plus what its pipeline writes. An IGSN with some of them but not all is partway through processing, or processing failed.

`xrd_calibrant_*` only applies to calibrant samples and `xrf_raw` is an optional second measurement on MAXIMA, so neither counts toward completeness. `nmd_derived` has no data anywhere yet, so it is left out too.

In [15]:
CORE_TYPES = {
    'HELIX': ['pdv_trace', 'pdv_experiment_log', 'pdv_alpss_output', 'pdv_alpss_result'],
    'MAXIMA': ['xrd_raw', 'xrd_metadata', 'xrd_derived', 'xrd_visualization'],
    'SPHINX': ['nmd_raw', 'nmd_project'],
}

rows = []
for instrument, types in CORE_TYPES.items():
    measured = has_type[types].any(axis=1)
    complete = has_type[types].all(axis=1)
    rows.append({
        'instrument': instrument,
        'IGSNs with any': int(measured.sum()),
        'complete': int(complete.sum()),
        'missing some': int((measured & ~complete).sum()),
        'nothing': int((~measured).sum()),
    })
pd.DataFrame(rows).set_index('instrument')

,IGSNs with any,complete,missing some,nothing
instrument,,,,
HELIX,370,321,49,143
MAXIMA,422,394,28,91
SPHINX,145,141,4,368


`nothing` counts IGSNs that have data from the other instruments but none from this one. The breakdown below names the missing types. The ones to chase are IGSNs with a raw type but no processed one (for example `pdv_trace` without `pdv_alpss_result`), and the reverse, where processed data or a log exists without the raw file.

## The catalog

Columns are grouped by instrument, and empty cells mean no partitions of that type. Rows are sorted so that IGSNs with data from the most instruments come first. Data types with nothing are dropped.

In [8]:
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 30)

live_types = [dt for dt in data_types if catalog[dt].any()]

table = catalog[live_types]
table.columns = pd.MultiIndex.from_tuples(
    [(instrument_of[dt], dt) for dt in live_types],
    names=['instrument', 'data type'],
)
table = table.sort_index(axis=1, level='instrument', sort_remaining=False)
table[('', 'instruments')] = summary['instruments']

order = summary.assign(n=by_instrument.sum(axis=1)).sort_values(['n', 'partitions'], ascending=False).index
dressed = table.loc[order].astype(str).replace('0', '')
dressed

instrument                    HELIX                                      \
data type          pdv_alpss_output pdv_alpss_result pdv_experiment_log   
JHABOX00000                       1                1                  2   
JHAMAL00022-007                   1                1                  2   
JHAMAL00016-068                   1                1                  2   
JHAMAL00021-017                   2                2                  2   
JHAMAL00016-025                   2                2                  2   
...                             ...              ...                ...   
TEST21                                                                    
TEST22                                                                    
TEST23                                                                    
JHAMAB00019-16                                                        1   
JHAMAL00020-S8R6C1                                                        

instrument                              MAXIMA                           \
data type          pdv_trace xrd_calibrant_raw xrd_derived xrd_metadata   
JHABOX00000                1                                              
JHAMAL00022-007            1                             4            4   
JHAMAL00016-068            1                             4            4   
JHAMAL00021-017            2                             3            3   
JHAMAL00016-025            2                             2            2   
...                      ...               ...         ...          ...   
TEST21                                                   1                
TEST22                                                   1                
TEST23                                                   1                
JHAMAB00019-16                                                            
JHAMAL00020-S8R6C1                                                        

instrument                                                SPHINX          \
data type          xrd_raw xrd_visualization xrf_raw nmd_project nmd_raw   
JHABOX00000                              809                           1   
JHAMAL00022-007          4                 4       2           2       2   
JHAMAL00016-068          4                 4       3           1       1   
JHAMAL00021-017          3                 3       2           2       2   
JHAMAL00016-025          2                 2       1           3       3   
...                    ...               ...     ...         ...     ...   
TEST21                   1                                                 
TEST22                   1                                                 
TEST23                   1                                                 
JHAMAB00019-16                                                             
JHAMAL00020-S8R6C1                                             1           

instrument                                   
data type                       instruments  
JHABOX00000         HELIX + MAXIMA + SPHINX  
JHAMAL00022-007     HELIX + MAXIMA + SPHINX  
JHAMAL00016-068     HELIX + MAXIMA + SPHINX  
JHAMAL00021-017     HELIX + MAXIMA + SPHINX  
JHAMAL00016-025     HELIX + MAXIMA + SPHINX  
...                                     ...  
TEST21                               MAXIMA  
TEST22                               MAXIMA  
TEST23                               MAXIMA  
JHAMAB00019-16                        HELIX  
JHAMAL00020-S8R6C1                   SPHINX  

[513 rows x 13 columns]

To look up one sample or family, filter on the IGSN prefix. Sub-samples (`-001`, `-S1R4C1`, …) are separate IGSNs with their own rows.

In [9]:
dressed[dressed.index.str.startswith('JHAMAL00016')]

instrument                 HELIX                                      \
data type       pdv_alpss_output pdv_alpss_result pdv_experiment_log   
JHAMAL00016-068                1                1                  2   
JHAMAL00016-025                2                2                  2   
JHAMAL00016-080                1                1                  1   
JHAMAL00016-082                1                1                  1   
JHAMAL00016-026                1                1                  1   
...                          ...              ...                ...   
JHAMAL00016-081                                                        
JHAMAL00016-083                                                        
JHAMAL00016-102                1                1                  1   
JHAMAL00016-001                1                1                      
JHAMAL00016-003                1                1                      

instrument                           MAXIMA                                   \
data type       pdv_trace xrd_calibrant_raw xrd_derived xrd_metadata xrd_raw   
JHAMAL00016-068         1                             4            4       4   
JHAMAL00016-025         2                             2            2       2   
JHAMAL00016-080         1                             3            3       3   
JHAMAL00016-082         1                             3            3       3   
JHAMAL00016-026         1                             2            2       2   
...                   ...               ...         ...          ...     ...   
JHAMAL00016-081                                       1            1       1   
JHAMAL00016-083                                       1            1       1   
JHAMAL00016-102         1                                                      
JHAMAL00016-001         1                                                      
JHAMAL00016-003         1                                                      

instrument                                     SPHINX          \
data type       xrd_visualization xrf_raw nmd_project nmd_raw   
JHAMAL00016-068                 4       3           1       1   
JHAMAL00016-025                 2       1           3       3   
JHAMAL00016-080                 3       2           1       1   
JHAMAL00016-082                 3       2           1       1   
JHAMAL00016-026                 2       1           3       3   
...                           ...     ...         ...     ...   
JHAMAL00016-081                 1                               
JHAMAL00016-083                 1                               
JHAMAL00016-102                                                 
JHAMAL00016-001                                                 
JHAMAL00016-003                                                 

instrument                                
data type                    instruments  
JHAMAL00016-068  HELIX + MAXIMA + SPHINX  
JHAMAL00016-025  HELIX + MAXIMA + SPHINX  
JHAMAL00016-080  HELIX + MAXIMA + SPHINX  
JHAMAL00016-082  HELIX + MAXIMA + SPHINX  
JHAMAL00016-026  HELIX + MAXIMA + SPHINX  
...                                  ...  
JHAMAL00016-081                   MAXIMA  
JHAMAL00016-083                   MAXIMA  
JHAMAL00016-102                    HELIX  
JHAMAL00016-001                    HELIX  
JHAMAL00016-003                    HELIX  

[82 rows x 13 columns]

## Sample coordinates

The coordinate-enrichment pipeline (`aimdl-coord-enrichment`) writes the position of each measurement onto its file:

- `Station_X` / `Station_Y`: where the instrument measured, in its own frame
- `Sample_X` / `Sample_Y`: the same point in the sample frame, which is comparable across instruments
- `coord_provenance`: which transform version and which station-side source produced them

`filters` does not allow `meta.Sample_X`, so the check is client-side. `limit=0` returns every item of a type in one request, and `extraFields` pulls the coordinate fields along. That is about 230k items, which takes under a minute.

`aimdl/count` reports more files for some types than `aimdl/datafiles` returns. The profile below is over what `datafiles` returns, the same set the rest of this notebook and the library work with.

In [10]:
COORD_FIELDS = ['meta.Sample_X', 'meta.Sample_Y', 'meta.Station_X', 'meta.Station_Y', 'meta.coord_provenance']

rows = []
with requests.Session() as session:
    gc = get_girder_client(session=session)
    for dt in live_types:
        items = gc.get('aimdl/datafiles', parameters={
            'dataType': dt,
            'limit': 0,
            'extraFields': json.dumps(COORD_FIELDS),
        })
        for item in items:
            meta = item['meta']
            prov = meta.get('coord_provenance') or {}
            rows.append({
                'data_type': dt,
                'igsn': meta.get('igsn'),
                'created': item['created'],
                'has_coords': 'Sample_X' in meta and 'Sample_Y' in meta,
                'transform': prov.get('transform_version'),
                'source': (prov.get('station_coord_source') or {}).get('kind'),
            })

coords = pd.DataFrame(rows)
coords['created'] = pd.to_datetime(coords['created'], format='ISO8601', utc=True)
print(f'{len(coords)} files, {coords["has_coords"].sum()} with sample coordinates')

247940 files, 156117 with sample coordinates


### Coverage by data type

Only some types are in scope for enrichment. Raw HELIX traces and MAXIMA scan points get coordinates from the station's own records (the experiment log, `instructions.txt`), and their derived files inherit them through `meta.prov.wasDerivedFrom`. Logs, metadata, calibrants and SPHINX files are not enriched, so 0% there is expected rather than a gap.

In [11]:
by_type = coords.groupby('data_type').agg(
    files=('has_coords', 'size'),
    with_coords=('has_coords', 'sum'),
)
by_type['without'] = by_type['files'] - by_type['with_coords']
by_type['coverage'] = (by_type['with_coords'] / by_type['files']).map('{:.0%}'.format)
by_type.insert(0, 'instrument', by_type.index.map(instrument_of))
by_type.sort_values(['instrument', 'files'], ascending=[True, False])

,instrument,files,with_coords,without,coverage
data_type,,,,,
pdv_alpss_output,HELIX,83189,46151,37038,55%
pdv_trace,HELIX,10899,5867,5032,54%
pdv_alpss_result,HELIX,10584,5769,4815,55%
pdv_experiment_log,HELIX,467,0,467,0%
xrd_raw,MAXIMA,48928,38022,10906,78%
xrd_derived,MAXIMA,40938,30421,10517,74%
xrd_visualization,MAXIMA,30854,19006,11848,62%
xrf_raw,MAXIMA,19668,10881,8787,55%
xrd_metadata,MAXIMA,1908,0,1908,0%


In [12]:
enriched_types = [dt for dt in live_types if by_type.loc[dt, 'with_coords'] > 0]

# Where the coordinates came from: transform version x station-side source
enriched = coords[coords['has_coords']]
pd.crosstab([enriched['data_type'].map(instrument_of), enriched['transform']], enriched['source'])

source               helix_experiment_log  inherited  maxima_instructions
data_type transform                                                      
HELIX     HELIX/v1                    965       8658                    0
          HELIX/v2                   4902      43262                    0
MAXIMA    MAXIMA/v1                     0          0                98330

### Which files are missing them

Within the enriched types, files without coordinates are either newer than the last enrichment sweep or ones the pipeline could not pair with a station record. Grouping by the month the file was uploaded tells the two apart. A backlog sits in the most recent months, and pairing failures are spread across older ones.

In [13]:
in_scope = coords[coords['data_type'].isin(enriched_types)]
missing = in_scope[~in_scope['has_coords']]

pd.crosstab(
    missing['created'].dt.strftime('%Y-%m'),
    missing['data_type'],
    margins=True,
).reindex(columns=[*[dt for dt in enriched_types if dt in missing['data_type'].unique()], 'All'])

data_type,pdv_alpss_output,pdv_alpss_result,pdv_trace,xrd_derived,xrd_raw,xrd_visualization,xrf_raw,All
created,,,,,,,,
2025-08,0,0,169,0,0,0,2,171
2025-09,0,0,0,0,2,0,277,279
2025-10,0,0,18,0,2,0,2846,2866
2025-11,0,0,3,0,0,0,395,398
2025-12,0,0,356,0,0,0,0,356
2026-01,0,0,37,0,0,0,0,37
2026-02,0,0,226,1,84,0,2043,2354
2026-03,0,0,49,10,342,0,1054,1455
2026-04,0,0,165,38,442,0,39,684


### Coverage by IGSN

For each IGSN and enriched type: **all** of its files have coordinates, **some** do, or **none** do. An IGSN under **none** cannot be placed on the sample at all for that measurement.

In [14]:
per_igsn = (
    in_scope.groupby(['igsn', 'data_type'])['has_coords']
    .agg(['sum', 'size'])
    .rename(columns={'sum': 'with', 'size': 'files'})
)
per_igsn['status'] = 'some'
per_igsn.loc[per_igsn['with'] == per_igsn['files'], 'status'] = 'all'
per_igsn.loc[per_igsn['with'] == 0, 'status'] = 'none'

(
    per_igsn.reset_index()
    .pivot_table(index='data_type', columns='status', values='files', aggfunc='size', fill_value=0)
    .reindex(index=enriched_types, columns=['all', 'some', 'none'], fill_value=0)
)

status,all,some,none
data_type,,,
pdv_alpss_output,187,38,136
pdv_alpss_result,187,38,136
pdv_trace,190,35,136
xrd_derived,292,12,115
xrd_raw,293,12,115
xrd_visualization,296,8,92
xrf_raw,164,11,139


### The coordinate catalog

One row per IGSN and one column per enriched type. Each cell shows `with/files`, blank when the IGSN has no files of that type. Rows are sorted with the least-covered IGSNs first, so the gaps are at the top.

In [15]:
cell = per_igsn['with'].astype(str) + '/' + per_igsn['files'].astype(str)
coord_table = cell.unstack('data_type').reindex(columns=enriched_types).fillna('')
coord_table.columns = pd.MultiIndex.from_tuples(
    [(instrument_of[dt], dt) for dt in enriched_types],
    names=['instrument', 'data type'],
)

igsn_totals = per_igsn.groupby('igsn')[['with', 'files']].sum()
coord_table[('', 'coverage')] = (igsn_totals['with'] / igsn_totals['files']).map('{:.0%}'.format)

order = (igsn_totals['with'] / igsn_totals['files']).sort_values(kind='stable').index
coord_table.loc[order]

instrument                     HELIX                                 MAXIMA  \
data type           pdv_alpss_output pdv_alpss_result pdv_trace xrd_derived   
igsn                                                                          
APLMEX00001                    0/175             0/25      0/25       0/100   
APLMEX00001-001                0/161             0/23      0/25       0/100   
APLMEX00001-002                0/105             0/15      0/19       0/100   
APLMEX00001-003                0/154             0/22      0/25       0/100   
APLMEX00001-004                0/161             0/23      0/25       0/100   
...                              ...              ...       ...         ...   
JHBMAI00003-S1R4C3                                                    24/24   
JHBMAI00003-S1R6C0                                                    24/24   
NWXMAB00009-002-001            24/24              3/3       3/3     170/170   
NWXMAB00010-001-001                                                 170/170   
NWXMAB00010-002-001            24/24              3/3       3/3     170/170   

instrument                                                       
data type            xrd_raw xrd_visualization xrf_raw coverage  
igsn                                                             
APLMEX00001            0/100              0/50    0/50       0%  
APLMEX00001-001        0/100              0/50    0/25       0%  
APLMEX00001-002        0/100              0/50    0/25       0%  
APLMEX00001-003        0/100              0/50    0/25       0%  
APLMEX00001-004        0/100              0/50    0/25       0%  
...                      ...               ...     ...      ...  
JHBMAI00003-S1R4C3     24/24             12/12   12/12     100%  
JHBMAI00003-S1R6C0     24/24             12/12   12/12     100%  
NWXMAB00009-002-001  170/170             85/85   85/85     100%  
NWXMAB00010-001-001  170/170             85/85   85/85     100%  
NWXMAB00010-002-001  170/170             85/85   85/85     100%  

[507 rows x 8 columns]

In [16]:
coord_table[coord_table.index.str.startswith('JHAMAL00016')]

instrument                 HELIX                                 MAXIMA  \
data type       pdv_alpss_output pdv_alpss_result pdv_trace xrd_derived   
igsn                                                                      
JHAMAL00016-001            0/192             0/24      0/24               
JHAMAL00016-002          568/600            71/75     71/75       50/50   
JHAMAL00016-003             0/48              0/6       0/6               
JHAMAL00016-004          152/352            19/44     19/44       50/50   
JHAMAL00016-005          192/392            24/49     24/49       50/50   
...                          ...              ...       ...         ...   
JHAMAL00016-081                                                   10/10   
JHAMAL00016-082          264/600            33/75     33/75     110/110   
JHAMAL00016-083                                                   10/10   
JHAMAL00016-084          200/200            25/25     25/25     110/110   
JHAMAL00016-102          168/216            21/27     21/27               

instrument                                                   
data type        xrd_raw xrd_visualization xrf_raw coverage  
igsn                                                         
JHAMAL00016-001                                          0%  
JHAMAL00016-002    50/50             25/25   25/25      96%  
JHAMAL00016-003                                          0%  
JHAMAL00016-004    50/50             25/25   25/25      58%  
JHAMAL00016-005    50/50             25/25   25/25      61%  
...                  ...               ...     ...      ...  
JHAMAL00016-081    10/10               5/5             100%  
JHAMAL00016-082  110/110             55/55   50/50      61%  
JHAMAL00016-083    10/10               5/5             100%  
JHAMAL00016-084  110/110             55/55   50/50     100%  
JHAMAL00016-102                                         78%  

[82 rows x 8 columns]